### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](https://media.npr.org/assets/img/2013/12/10/istock-18586699-monkey-computer_brick-16e5064d3378a14e0e4c2da08857efe03c04695e-s800-c85.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
!wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

,author,day,id,link,month,summary,tag,title,year
13509,"[{'name': 'Rizwana Kalsoom'}, {'name': 'Moomal...",7,1406.2614v4,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",6,This paper has been withdrawn by the author du...,"[{'term': 'cs.NE', 'scheme': 'http://arxiv.org...",Application and Verification of Algorithm Lear...,2014
23921,[{'name': 'Igor Polkovnikov'}],20,1303.4840v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,A variety of operations of cellular automata o...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Asynchronous Cellular Operations on Gray Image...,2013
34125,"[{'name': 'John David Osborne'}, {'name': 'Bin...",7,1402.1668v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",2,We used MetaMap and YTEX as a basis for the co...,"[{'term': 'cs.IR', 'scheme': 'http://arxiv.org...",Evaluation of YTEX and MetaMap for clinical co...,2014
39309,"[{'name': 'Irina A. Kogan'}, {'name': 'Peter J...",22,1509.06690v1,"[{'rel': 'related', 'href': 'http://dx.doi.org...",9,We examine the relationships between the diffe...,"[{'term': 'math.DG', 'scheme': 'http://arxiv.o...",Invariants of objects and their images under s...,2015
1157,"[{'name': 'Xichuan Zhou'}, {'name': 'Shengli L...",21,1604.06154v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",4,Deep neural networks are state-of-the-art mode...,"[{'term': 'cs.LG', 'scheme': 'http://arxiv.org...",Deep Adaptive Network: An Efficient Deep Neura...,2016


In [6]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [7]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer

from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()

# Convert each line to lowercase and tokenize, then join tokens with spaces
lines = [' '.join(tokenizer.tokenize(line.lower())) for line in lines]

In [5]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

In [6]:
# Let's see a few examples of the tokenized lines
print("Sample tokenized lines:")
for i in range(3):
    print(f"{i+1}: {sorted(lines, key=len)[i]}")
    
print(f"\nTotal number of lines: {len(lines)}")
print(f"Average tokens per line: {sum(len(line.split()) for line in lines) / len(lines):.1f}")

Sample tokenized lines:
1: differential contrastive divergence ; this paper has been retracted .
2: what does artificial life tell us about death ? ; short philosophical essay
3: p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .

Total number of lines: 41000
Average tokens per line: 188.3


### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:
$$
 P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1}). 
$$

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [8]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens: 
# - `UNK` represents absent tokens, 
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    counts = defaultdict(Counter)

    for line in tqdm(lines):
        tokens = line.split() + [EOS]

        for i in range(len(tokens)):
            target_token = tokens[i]

            prefix_start = max(0, i - n + 1)
            prefix = tokens[prefix_start:i]

            while len(prefix) < n - 1:
                prefix = [UNK] + prefix

            prefix_tuple = tuple(prefix)
            counts[prefix_tuple][target_token] += 1
    
    return counts

In [9]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

100%|██████████| 100/100 [00:00<00:00, 26792.10it/s]


Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [10]:
class NGramLanguageModel:    
    def __init__(self, lines, n):
        """ 
        Train a simple count-based language model: 
        compute probabilities P(w_t | prefix) given ngram counts
        
        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n
    
        counts = count_ngrams(lines, self.n)

        self.probs = defaultdict(Counter)

        for prefix, token_counts in counts.items():
            total_count = sum(token_counts.values())
            for token, count in token_counts.items():
                self.probs[prefix][token] = count / total_count
            
    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]
    
    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [10]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

100%|██████████| 100/100 [00:00<00:00, 14857.61it/s]


Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [11]:
lm = NGramLanguageModel(lines, n=3)

100%|██████████| 41000/41000 [00:12<00:00, 3375.35it/s]



The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [11]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    token_probs = lm.get_possible_next_tokens(prefix)
    
    if len(token_probs) == 0:
        return EOS
    
    tokens = list(token_probs.keys())
    probs = list(token_probs.values())
    
    if temperature == 0.0:
        max_prob = max(probs)
        best_tokens = [token for token, prob in token_probs.items() if prob == max_prob]
        return np.random.choice(best_tokens)
    else:
        probs = np.array(probs)
        probs = probs ** (1.0 / temperature)
        probs = probs / np.sum(probs)

        return np.random.choice(tokens, p=probs)

In [13]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [14]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

artificial prediction markets also allow discrete deterministic information when generates and evaluates a subset of maps was greatly simplified , sub - word relatedness that combines sparse coding ( gsc ), which allows learning from real images that were established and applied to the boltzmann machine , and is as low - precision data representation problems . _EOS_


In [15]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

bridging the gap between the two - dimensional vector space model . we also demonstrate that our approach is based on the other hand , we first introduce a novel approach for the first time , the proposed method is proposed . the feature vector and a set of features . these results are presented . the system . in order to improve the results . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $1/N$, where $N$ is __total length (in tokens) of all sentences__ in corpora.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [12]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above
    
    Note: do not forget to compute P(w_first | empty) and P(eos | full_sequence)
    
    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    total_log_prob = 0.0
    total_tokens = 0
    
    for line in lines:
        tokens = line.split() + [EOS]
        
        for i in range(len(tokens)):
            prefix_tokens = tokens[:i]
            prefix = ' '.join(prefix_tokens)
            current_token = tokens[i]

            prob = lm.get_next_token_prob(prefix, current_token)

            log_prob = np.log(prob) if prob > 0 else min_logprob
            log_prob = max(log_prob, min_logprob)
            
            total_log_prob += log_prob
            total_tokens += 1

    avg_log_prob = total_log_prob / total_tokens
    perplexity_value = np.exp(-avg_log_prob)
    
    return perplexity_value

In [17]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

100%|██████████| 100/100 [00:00<00:00, 14485.59it/s]

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [ ]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


100%|██████████| 30750/30750 [00:02<00:00, 11079.69it/s]



N = 1, Perplexity = 1832.23136


100%|██████████| 30750/30750 [00:05<00:00, 5725.02it/s]



N = 2, Perplexity = 85653987.28774


100%|██████████| 30750/30750 [00:09<00:00, 3350.98it/s]



N = 3, Perplexity = 61999196259043346743296.00000


In [ ]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [19]:
class LaplaceLanguageModel(NGramLanguageModel): 
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}
    
    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total)
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        

**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [20]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

100%|██████████| 100/100 [00:00<00:00, 26318.03it/s]



In [21]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|██████████| 30750/30750 [00:02<00:00, 11376.68it/s]



N = 1, Perplexity = 1832.66878


100%|██████████| 30750/30750 [00:04<00:00, 6426.46it/s]



N = 2, Perplexity = 470.48021


100%|██████████| 30750/30750 [00:08<00:00, 3446.54it/s]



N = 3, Perplexity = 3679.44765


In [ ]:
# optional: try to sample tokens from such a model


In [23]:
# Давайте проанализируем, почему сглаживание помогает
print("=== Анализ проблемы оригинальной модели ===")

# Попробуем понять, почему оригинальная модель застревает
original_lm = NGramLanguageModel(train_lines, n=3)
smoothed_lm = LaplaceLanguageModel(train_lines, n=3, delta=0.1)

test_phrases = ['machine learning', 'deep neural', 'artificial intelligence']

for phrase in test_phrases:
    print(f"\nТестируем фразу: '{phrase}'")
    
    # Оригинальная модель
    orig_tokens = original_lm.get_possible_next_tokens(phrase)
    print(f"  Оригинальная модель: {len(orig_tokens)} возможных токенов")
    if len(orig_tokens) > 0:
        top_orig = sorted(orig_tokens.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"    Топ-3: {top_orig}")
    
    # Модель со сглаживанием
    smooth_tokens = smoothed_lm.get_possible_next_tokens(phrase)
    print(f"  Модель со сглаживанием: {len(smooth_tokens)} возможных токенов")
    if len(smooth_tokens) > 0:
        top_smooth = sorted(smooth_tokens.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"    Топ-3: {top_smooth}")

print(f"\n=== Размер словаря ===")
print(f"Размер словаря модели со сглаживанием: {len(smoothed_lm.vocab)} токенов")
print(f"Это гарантирует, что модель всегда найдет следующий токен!")

=== Анализ проблемы оригинальной модели ===


100%|██████████| 30750/30750 [00:08<00:00, 3473.17it/s]



Тестируем фразу: 'machine learning'
  Оригинальная модель: 319 возможных токенов
    Топ-3: [('.', 0.08632019115890084), ('algorithms', 0.07407407407407407), ('and', 0.06929510155316607)]
  Модель со сглаживанием: 54176 возможных токенов
    Топ-3: [('.', 0.0329811992333668), ('algorithms', 0.02830382403942685), ('and', 0.026478506890572236)]

Тестируем фразу: 'deep neural'
  Оригинальная модель: 22 возможных токенов
    Топ-3: [('networks', 0.6529933481152993), ('network', 0.307649667405765), ('nets', 0.01164079822616408)]
  Модель со сглаживанием: 54176 возможных токенов
    Топ-3: [('networks', 0.16313559322033896), ('network', 0.07686662235515675), ('nets', 0.002921790185000554)]

Тестируем фразу: 'artificial intelligence'
  Оригинальная модель: 115 возможных токенов
    Топ-3: [('(', 0.1737012987012987), ('.', 0.1444805194805195), (',', 0.1266233766233766)]
  Модель со сглаживанием: 54176 возможных токенов
    Топ-3: [('(', 0.017750596658711214), ('.', 0.014767303102625296), (','

### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1
- Unigram $P_{kn}(w_t | prefix_{n-2})$ corresponds to Kneser Ney smoothing for {N-1}-gram language model.
- Unigram $P_{kn}(w_t)$ is a special case: how likely it is to see x_t in an unfamiliar context

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

In [30]:
from functools import lru_cache

def count_all_ngrams(lines, max_n):
    counts = {i: defaultdict(Counter) for i in range(1, max_n + 1)}
    for line in lines:
        tokens = line.split() + [EOS]
        for i in range(len(tokens)):
            for n in range(1, max_n + 1):
                start = max(0, i - n + 1)
                prefix = tokens[start:i]
                while len(prefix) < n - 1:
                    prefix = [UNK] + prefix
                counts[n][tuple(prefix)][tokens[i]] += 1
    return counts


class KneserNeyLanguageModel:
    def __init__(self, lines, n=3, delta=0.75):
        self.n = n
        self.delta = delta
        self.counts_by_order = count_all_ngrams(lines, n)
        vocab = set()
        for counts in self.counts_by_order.values():
            for prefix_counts in counts.values():
                vocab.update(prefix_counts.keys())
        self.vocab = list(vocab)
        if n >= 2:
            self.continuation_probs = self._compute_continuation_probs(self.counts_by_order[2])
        else:
            self.continuation_probs = {w: 1 / len(self.vocab) for w in self.vocab}
        self._get_lower_prob_cached = lru_cache(maxsize=200_000)(self._get_lower_prob)
        self.get_next_token_prob = lru_cache(maxsize=1_000_000)(self.get_next_token_prob)

    def _compute_continuation_probs(self, bigram_counts):
        continuation = Counter()
        for prefix, token_counts in bigram_counts.items():
            for token in token_counts:
                continuation[token] += 1
        total = sum(continuation.values()) or 1
        return {w: c / total for w, c in continuation.items()}

    def _get_lower_prob(self, prefix, token):
        if len(prefix) == 0 or len(prefix) + 1 not in self.counts_by_order:
            return self.continuation_probs.get(token, 1 / len(self.vocab))
        lower_prefix = tuple(prefix[1:])
        level = len(prefix)
        lower_counts = self.counts_by_order[level].get(lower_prefix, {})
        total = sum(lower_counts.values())
        if total == 0:
            return self.continuation_probs.get(token, 1 / len(self.vocab))
        count = lower_counts.get(token, 0)
        discounted = max(count - self.delta, 0) / total
        lambda_mass = (self.delta * len(lower_counts)) / total
        return discounted + lambda_mass * self._get_lower_prob_cached(lower_prefix, token)

    def get_next_token_prob(self, prefix, token):
        tokens = prefix.split() if prefix else []
        tokens = tokens[-(self.n - 1):]
        key = tuple(tokens)
        level = len(key) + 1
        if level not in self.counts_by_order:
            return self.continuation_probs.get(token, 1 / len(self.vocab))
        counts = self.counts_by_order[level].get(key, {})
        total = sum(counts.values())
        if total == 0:
            return self._get_lower_prob_cached(key, token)
        count = counts.get(token, 0)
        discounted = max(count - self.delta, 0) / total
        lambda_mass = (self.delta * len(counts)) / total
        backoff = self._get_lower_prob_cached(key, token)
        return discounted + lambda_mass * backoff

    def get_possible_next_tokens(self, prefix):
        tokens = prefix.split() if prefix else []
        key = tuple(tokens[-(self.n - 1):])
        level = len(key) + 1
        counts = self.counts_by_order.get(level, {}).get(key, {})
        total = sum(counts.values())
        if total == 0:
            return self.continuation_probs
        lambda_mass = (self.delta * len(counts)) / total
        probs = {}
        for t, c in counts.items():
            discounted = max(c - self.delta, 0) / total
            backoff = self._get_lower_prob_cached(key, t)
            probs[t] = discounted + lambda_mass * backoff
        s = sum(probs.values())
        if s > 0:
            for t in probs:
                probs[t] /= s
        return probs


In [31]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [32]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

N = 1, Perplexity = 53516.83226
N = 2, Perplexity = 201.25176
N = 3, Perplexity = 152.98043


In [ ]:
lm = KneserNeyLanguageModel(train_lines, n=3)

In [36]:
prefix = 'once'
for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.7)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

once . we show that our proposed method is a declarative model for a model that simultaneously learns to predict the next frame . we evaluate these using a linear system identification , and the nature of the population , it requires only a single image super - resolution image data and model - based on a number of distinct values . this paper presents a novel method for learning and the bayesian decision problems with dependent arms . we demonstrate that our model , we first introduce a new dataset for sequential tagging . our proposed method on the challenging
